Importing and converting "CRIME" data to .csv to work with it

In [ ]:
pip install pyaxis

CHANGING DIRECTORY SO THAT WE'RE IN THE RIGHT PLACE

In [1]:

%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/DATA"
%pwd

/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/DATA


'/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/DATA'

Reading .px file

In [2]:
from pyaxis import pyaxis

data = pyaxis.parse("Data_Crime_messy", encoding="utf-8-sig") #reading .PX file --> "encoding=..." necessary bc required by pyaxis

Multilingual PX file


In [ ]:
data
#df = data['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

#df.to_csv('Crime_clean.csv', index=False) #index = False --> so that not 0, 1, .... row lables are created



In [ ]:
import pandas as pd

#crimeClean = pd.read_csv("Crime_clean.csv")

In [ ]:
#crimeClean.columns #looking if it imported everything correctly

Importing and converting socioeconomic data to .csv to work with it

In [ ]:
#pop = pyaxis.parse("Data_Population_messy", encoding="utf-8-sig")

#pop
#dfp = pop['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

#dfp.to_csv('Population_clean.parquet', index=False)

#PopClean = pd.read_parquet("Population_clean.parquet")

#PopClean.columns

Trying with Parquet bc .csv was too huge

In [3]:
import sys
!{sys.executable} -m pip install pyarrow #pip command is not working and I cannot fix--> workaround


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: pip3.12 install --upgrade pip


In [4]:
from pyaxis import pyaxis 
import pandas as pd
import pyarrow


pop = pyaxis.parse("Data_Population_messy", encoding="utf-8-sig")

pop
dfp = pop['DATA'] #creating dataframe out of dictionary so that we can actually work with the data inside DATA folder

dfp.to_parquet('Population_clean.parquet', index=False)

PopClean = pd.read_parquet("Population_clean.parquet")

print(PopClean.columns)
print(PopClean.head())

Multilingual PX file
Index(['Jahr', 'Kanton (-) / Bezirk (>>) / Gemeinde (......)',
       'Bevölkerungstyp', 'Staatsangehörigkeit (Kategorie)', 'Geschlecht',
       'Alter', 'DATA'],
      dtype='str')
   Jahr Kanton (-) / Bezirk (>>) / Gemeinde (......)  \
0  2010                                      Schweiz   
1  2010                                      Schweiz   
2  2010                                      Schweiz   
3  2010                                      Schweiz   
4  2010                                      Schweiz   

            Bevölkerungstyp          Staatsangehörigkeit (Kategorie)  \
0  Ständige Wohnbevölkerung  Staatsangehörigkeit (Kategorie) - Total   
1  Ständige Wohnbevölkerung  Staatsangehörigkeit (Kategorie) - Total   
2  Ständige Wohnbevölkerung  Staatsangehörigkeit (Kategorie) - Total   
3  Ständige Wohnbevölkerung  Staatsangehörigkeit (Kategorie) - Total   
4  Ständige Wohnbevölkerung  Staatsangehörigkeit (Kategorie) - Total   

           Geschlecht      

Dropping unnecessary variables so that the file gets smaller

In [ ]:
#PopClean = PopClean.drop(columns=['Kanton (-)', 'Gemeinde (......)'])

In [ ]:
#print(PopClean[PopClean['Kanton (-) / Bezirk (>>) / Gemeinde (......)'].str.contains('Wettswil', na=False)].head())

values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #show me unique values to see how I need to adjust it
print(values)

In [5]:
#dropping all Gemeinde and Bezirk data (granular level we do not need)

PopClean = PopClean[~PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.contains(r">>|\.\.\.\.\.\.", na=False)]
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique()
print(values)

#"~" inverts boolean condition, so that PopClean contains only rows that do not contain the strings listed in the command

<ArrowStringArray>
[                           'Schweiz',                           '- Zürich',
                     '- Bern / Berne',                           '- Luzern',
                              '- Uri',                           '- Schwyz',
                         '- Obwalden',                        '- Nidwalden',
                           '- Glarus',                              '- Zug',
              '- Fribourg / Freiburg',                        '- Solothurn',
                      '- Basel-Stadt',                 '- Basel-Landschaft',
                     '- Schaffhausen',           '- Appenzell Ausserrhoden',
            '- Appenzell Innerrhoden',                       '- St. Gallen',
 '- Graubünden / Grigioni / Grischun',                           '- Aargau',
                          '- Thurgau',                           '- Ticino',
                             '- Vaud',                  '- Valais / Wallis',
                        '- Neuchâtel',                   

Cleaning up the values so that they match the values of cantons in the Crime dataset

In [6]:
#cleaning so that we only keep the names of the cantons in their language (first term)
PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.split('/').str[0] #splitting and then keeping only the [0] value
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

['Schweiz' '- Zürich' '- Bern ' '- Luzern' '- Uri' '- Schwyz' '- Obwalden'
 '- Nidwalden' '- Glarus' '- Zug' '- Fribourg ' '- Solothurn'
 '- Basel-Stadt' '- Basel-Landschaft' '- Schaffhausen'
 '- Appenzell Ausserrhoden' '- Appenzell Innerrhoden' '- St. Gallen'
 '- Graubünden ' '- Aargau' '- Thurgau' '- Ticino' '- Vaud' '- Valais '
 '- Neuchâtel' '- Genève' '- Jura' 'Ohne Angabe']


In [7]:
#stripping values of the whitespaces and the "-"

PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.replace('-', '', regex=False) #regex=False --> look at exact character

PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].str.strip()

values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

['Schweiz' 'Zürich' 'Bern' 'Luzern' 'Uri' 'Schwyz' 'Obwalden' 'Nidwalden'
 'Glarus' 'Zug' 'Fribourg' 'Solothurn' 'BaselStadt' 'BaselLandschaft'
 'Schaffhausen' 'Appenzell Ausserrhoden' 'Appenzell Innerrhoden'
 'St. Gallen' 'Graubünden' 'Aargau' 'Thurgau' 'Ticino' 'Vaud' 'Valais'
 'Neuchâtel' 'Genève' 'Jura' 'Ohne Angabe']


In [8]:
#dropping all rows with "Ohne Angabe" because if we do not get a canton the values are useless to us

PopClean = PopClean[PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"] != "Ohne Angabe"]
values = PopClean["Kanton (-) / Bezirk (>>) / Gemeinde (......)"].unique() #see if it worked
print(values)

#and renaming the the column name

PopClean = PopClean.rename(columns={"Kanton (-) / Bezirk (>>) / Gemeinde (......)": "Kanton"})

['Schweiz' 'Zürich' 'Bern' 'Luzern' 'Uri' 'Schwyz' 'Obwalden' 'Nidwalden'
 'Glarus' 'Zug' 'Fribourg' 'Solothurn' 'BaselStadt' 'BaselLandschaft'
 'Schaffhausen' 'Appenzell Ausserrhoden' 'Appenzell Innerrhoden'
 'St. Gallen' 'Graubünden' 'Aargau' 'Thurgau' 'Ticino' 'Vaud' 'Valais'
 'Neuchâtel' 'Genève' 'Jura']


Cleaning the file so that the ages match

In [9]:
#rename column so that it is easier to work with:
PopClean["Alter"] = PopClean["Alter"].str.strip() #stripping of all whitespaces
PopClean["Alter"] = PopClean["Alter"].str.split(' ').str[0] #only taking first value

#dropping all data that is not 19<=age<=29
PopClean["Alter"] = pd.to_numeric(PopClean["Alter"], errors="coerce")
PopClean = PopClean[PopClean["Alter"] >= 20]
PopClean = PopClean[PopClean["Alter"] <= 29]

#test if it worked:
values = PopClean["Alter"].unique() #see if it worked
print(values)
print(PopClean.columns)
print(PopClean.head())

[20. 21. 22. 23. 24. 25. 26. 27. 28. 29.]
Index(['Jahr', 'Kanton', 'Bevölkerungstyp', 'Staatsangehörigkeit (Kategorie)',
       'Geschlecht', 'Alter', 'DATA'],
      dtype='str')
    Jahr   Kanton           Bevölkerungstyp  \
21  2010  Schweiz  Ständige Wohnbevölkerung   
22  2010  Schweiz  Ständige Wohnbevölkerung   
23  2010  Schweiz  Ständige Wohnbevölkerung   
24  2010  Schweiz  Ständige Wohnbevölkerung   
25  2010  Schweiz  Ständige Wohnbevölkerung   

            Staatsangehörigkeit (Kategorie)          Geschlecht  Alter   DATA  
21  Staatsangehörigkeit (Kategorie) - Total  Geschlecht - Total   20.0  96146  
22  Staatsangehörigkeit (Kategorie) - Total  Geschlecht - Total   21.0  95916  
23  Staatsangehörigkeit (Kategorie) - Total  Geschlecht - Total   22.0  97690  
24  Staatsangehörigkeit (Kategorie) - Total  Geschlecht - Total   23.0  96329  
25  Staatsangehörigkeit (Kategorie) - Total  Geschlecht - Total   24.0  98900  


In [16]:
#filtering for type of visa
print(PopClean["Bevölkerungstyp"].unique()) #checking what current values are

PopClean = PopClean[PopClean["Bevölkerungstyp"] == "Ständige Wohnbevölkerung"] #including only permanent Stays
print(PopClean["Bevölkerungstyp"].unique())


#wanting to drop "Staatsangehörigkeit"

staat = PopClean["Staatsangehörigkeit (Kategorie)"].unique() #see if it worked
print(staat) 
#here we can see that we have "swiss, foreign and total" 

#--> we should only keep "Total"
PopClean = PopClean[PopClean['Staatsangehörigkeit (Kategorie)'] == 'Staatsangehörigkeit (Kategorie) - Total']

#now dropping collumn "Staatsangehörigkeit" because it is a granular level we don't need
PopClean = PopClean.drop(columns=["Staatsangehörigkeit (Kategorie)"])


print(values)
print(PopClean.columns)
print(PopClean.head())

<ArrowStringArray>
['Ständige Wohnbevölkerung']
Length: 1, dtype: str
<ArrowStringArray>
['Ständige Wohnbevölkerung']
Length: 1, dtype: str
<ArrowStringArray>
['Staatsangehörigkeit (Kategorie) - Total']
Length: 1, dtype: str
[20. 21. 22. 23. 24. 25. 26. 27. 28. 29.]
Index(['Jahr', 'Kanton', 'Bevölkerungstyp', 'Geschlecht', 'Alter', 'DATA'],
      dtype='str')
    Jahr   Kanton           Bevölkerungstyp          Geschlecht  Alter   DATA
21  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   20.0  96146
22  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   21.0  95916
23  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   22.0  97690
24  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   23.0  96329
25  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   24.0  98900


Renaming cantons so that they will match the other dataset

In [18]:
#looking exactly at what we have in current working file
values = PopClean["Kanton"].unique() #see if it worked
print(values)
print(PopClean.columns)
print(PopClean.head())

#looking at what Caner did for the Crime file
%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wage/Group_project_part_caner"
%pwd

crime = pd.read_csv("df_B_20-29_kombiniert_clean.csv")


cvalues = crime["Kanton"].unique() #see if it worked
print(cvalues)
print(crime.columns)
print(crime.head())

#dropping cantons that are not in the Crime dataset and changing the names so that they match
PopClean = PopClean[~PopClean["Kanton"].isin(["Uri", "Schwyz", "Obwalden", "Nidwalden", "Glarus", "Zug", "Schaffhausen", "Appenzell Ausserrhoden", "Appenzell Innerrhoden", "Graubünden", "Ticino", "Jura"])] #.isin checks if canton is in list --> ~ reverses condition

#now renaming values so that they match
PopClean["Kanton"] = PopClean["Kanton"].replace({"BaselLandschaft": "Basel-Landschaft", "BaselStadt": "Basel-Stadt", "Fribourg": "Freiburg", "Genève": "Genf", "Neuchâtel": "Neuenburg", "Vaud": "Waadt", "Valais": "Wallis"})


#see if it worked:
values = PopClean["Kanton"].unique() #see if it worked
print(values)
cvalues = crime["Kanton"].unique() #see if it worked
print(cvalues)


['Schweiz' 'Zürich' 'Bern' 'Luzern' 'Uri' 'Schwyz' 'Obwalden' 'Nidwalden'
 'Glarus' 'Zug' 'Fribourg' 'Solothurn' 'BaselStadt' 'BaselLandschaft'
 'Schaffhausen' 'Appenzell Ausserrhoden' 'Appenzell Innerrhoden'
 'St. Gallen' 'Graubünden' 'Aargau' 'Thurgau' 'Ticino' 'Vaud' 'Valais'
 'Neuchâtel' 'Genève' 'Jura']
Index(['Jahr', 'Kanton', 'Bevölkerungstyp', 'Geschlecht', 'Alter', 'DATA'],
      dtype='str')
    Jahr   Kanton           Bevölkerungstyp          Geschlecht  Alter   DATA
21  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   20.0  96146
22  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   21.0  95916
23  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   22.0  97690
24  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   23.0  96329
25  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   24.0  98900
/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wag

In [19]:
print(PopClean.head())
print(crime.head())

gender = PopClean["Geschlecht"].unique() #see if it worked
print(gender)

    Jahr   Kanton           Bevölkerungstyp          Geschlecht  Alter   DATA
21  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   20.0  96146
22  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   21.0  95916
23  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   22.0  97690
24  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   23.0  96329
25  2010  Schweiz  Ständige Wohnbevölkerung  Geschlecht - Total   24.0  98900
   Kanton  Jahr Altersklasse  Beschuldigte
0  Aargau  2009        20-29          82.0
1  Aargau  2010        20-29          78.0
2  Aargau  2011        20-29          78.0
3  Aargau  2012        20-29         104.0
4  Aargau  2013        20-29         100.0
<ArrowStringArray>
['Geschlecht - Total', 'Mann', 'Frau']
Length: 3, dtype: str


In [20]:
#grouping for ages across 20-29
PopClean['DATA'] = pd.to_numeric(PopClean['DATA'], errors='coerce')

PopClean = PopClean.groupby(['Jahr', 'Kanton', 'Bevölkerungstyp', 'Geschlecht'], as_index=False)['DATA'].sum()

print(PopClean.head())

Clean_genderless = PopClean[PopClean["Geschlecht"] == "Geschlecht - Total"] #only looking at "genderless" data
Clean_genderless = Clean_genderless.drop(columns=["Geschlecht"])

print(Clean_genderless.head())

   Jahr            Kanton           Bevölkerungstyp          Geschlecht   DATA
0  2010            Aargau  Ständige Wohnbevölkerung                Frau  38865
1  2010            Aargau  Ständige Wohnbevölkerung  Geschlecht - Total  79369
2  2010            Aargau  Ständige Wohnbevölkerung                Mann  40504
3  2010  Basel-Landschaft  Ständige Wohnbevölkerung                Frau  15134
4  2010  Basel-Landschaft  Ständige Wohnbevölkerung  Geschlecht - Total  30893
    Jahr            Kanton           Bevölkerungstyp    DATA
1   2010            Aargau  Ständige Wohnbevölkerung   79369
4   2010  Basel-Landschaft  Ständige Wohnbevölkerung   30893
7   2010       Basel-Stadt  Ständige Wohnbevölkerung   24488
10  2010              Bern  Ständige Wohnbevölkerung  120344
13  2010          Freiburg  Ständige Wohnbevölkerung   36620


In [21]:
#looking at headers to see how I need to change the structure of the dataset for merging

print(Clean_genderless.head())
print(crime.head())

    Jahr            Kanton           Bevölkerungstyp    DATA
1   2010            Aargau  Ständige Wohnbevölkerung   79369
4   2010  Basel-Landschaft  Ständige Wohnbevölkerung   30893
7   2010       Basel-Stadt  Ständige Wohnbevölkerung   24488
10  2010              Bern  Ständige Wohnbevölkerung  120344
13  2010          Freiburg  Ständige Wohnbevölkerung   36620
   Kanton  Jahr Altersklasse  Beschuldigte
0  Aargau  2009        20-29          82.0
1  Aargau  2010        20-29          78.0
2  Aargau  2011        20-29          78.0
3  Aargau  2012        20-29         104.0
4  Aargau  2013        20-29         100.0


Relative Population numbers

In [22]:
Clean_genderless = Clean_genderless[['Kanton', 'Jahr', 'Bevölkerungstyp', 'DATA']]

#create new dataset for swiss data
data_swiss = Clean_genderless[Clean_genderless["Kanton"] == "Schweiz"][["Jahr", "Bevölkerungstyp", "DATA"]].rename(columns={"DATA": "DATA_Schweiz"})

#merging new subset with old dataset
Clean_genderless = Clean_genderless.merge(data_swiss, on=["Jahr", "Bevölkerungstyp"], how="left")
Clean_genderless["Percentage"] = (Clean_genderless["DATA"] / Clean_genderless["DATA_Schweiz"]) * 100


print(Clean_genderless.head())


             Kanton  Jahr           Bevölkerungstyp    DATA  DATA_Schweiz  \
0            Aargau  2010  Ständige Wohnbevölkerung   79369       1006333   
1  Basel-Landschaft  2010  Ständige Wohnbevölkerung   30893       1006333   
2       Basel-Stadt  2010  Ständige Wohnbevölkerung   24488       1006333   
3              Bern  2010  Ständige Wohnbevölkerung  120344       1006333   
4          Freiburg  2010  Ständige Wohnbevölkerung   36620       1006333   

   Percentage  
0    7.886952  
1    3.069859  
2    2.433389  
3   11.958666  
4    3.638955  


In [23]:
#converting both "Jahr" to numeric figures
crime["Jahr"] = pd.to_numeric(crime["Jahr"], errors="coerce").astype("Int64")
Clean_genderless["Jahr"] = pd.to_numeric(Clean_genderless["Jahr"], errors="coerce").astype("Int64")

#merging
merged_df = pd.merge(Clean_genderless, crime, on=["Kanton", "Jahr"], how="inner") #inner only keeping exact matches of combinations

print(merged_df.head())

             Kanton  Jahr           Bevölkerungstyp    DATA  DATA_Schweiz  \
0            Aargau  2010  Ständige Wohnbevölkerung   79369       1006333   
1  Basel-Landschaft  2010  Ständige Wohnbevölkerung   30893       1006333   
2       Basel-Stadt  2010  Ständige Wohnbevölkerung   24488       1006333   
3              Bern  2010  Ständige Wohnbevölkerung  120344       1006333   
4          Freiburg  2010  Ständige Wohnbevölkerung   36620       1006333   

   Percentage Altersklasse  Beschuldigte  
0    7.886952        20-29          78.0  
1    3.069859        20-29          16.0  
2    2.433389        20-29          86.0  
3   11.958666        20-29         260.0  
4    3.638955        20-29          12.0  


SAVING WHAT I DID IN THE CLEAN DATA FILE

In [25]:
%cd "/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wage/Group_project_part_annette/Data_cleaning_and_merging"
%pwd
PopClean.to_csv("Population_clean.csv", index=False)
Clean_genderless.to_csv("Population_clean_without_gender.csv", index=False)
merged_df.to_csv("Crime_Pop_merged.csv", index=False)

/Users/annetteschaudt/Documents/UZH shit/Data Analysis with Python/DAwP Project/Python-Project-HS26-Criminality-and-Minimum-Wage/Group_project_part_annette/Data_cleaning_and_merging
